# U09 | 中文数据处理：构建中英训练 batch

**目标**：把 U08 的玩具数据（数字反转）换成真实的中英平行语料，跑通从「原始文本 → 训练 batch」的完整流水线。

**过关标准**：能独立从一份 `.txt` 平行语料生成 `(src, src_len, tgt)` 的训练 batch，shape 正确，pad/SOS/EOS 都对。

## 本单元会回答
1. 真实翻译数据长什么样？
2. 中文怎么分词？英文怎么分词？为什么不一样？
3. 怎么把字符串变成 GPU 能算的 `LongTensor`？
4. 一个 batch 里句子长度不一样怎么办？
5. 为什么要记录每条句子的真实长度？
6. 完整的 `Dataset` + `DataLoader` 怎么搭？

## 1. 真实翻译数据长什么样

标准的平行语料是「一行中文 + 一行英文」或「中文\t英文」的对齐格式：

```
我爱你。\tI love you.
今天天气真好。\tThe weather is nice today.
他正在学习深度学习。\tHe is learning deep learning.
```

常见公开数据集：
- **WMT**（学术翻译评测，规模大）
- **OpenSubtitles**（电影字幕，口语化）
- **news-commentary**（新闻评论，正式）
- **Tatoeba**（短句，适合入门）

本课用一份**手工小语料**演示流水线，跑通后再换大数据集。

In [ ]:
# 准备一份小语料（直接写在内存里，避免下载）
raw_pairs = [
    ('我爱你', 'I love you'),
    ('我喜欢猫', 'I like cats'),
    ('他在看书', 'He is reading a book'),
    ('今天天气真好', 'The weather is nice today'),
    ('她正在学习深度学习', 'She is learning deep learning'),
    ('我们一起去公园', 'Let us go to the park'),
    ('这本书很有趣', 'This book is interesting'),
    ('你叫什么名字', 'What is your name'),
    ('我来自中国', 'I am from China'),
    ('明天见', 'See you tomorrow'),
]

for zh, en in raw_pairs[:3]:
    print(f'{zh} | {en}')

## 2. 分词（tokenization）

**为什么要分词**：模型只认识数字 ID，不认识字符串。`'我爱你'` 必须先切成 token 列表 `['我', '爱', '你']`，才能查表变成 `[5, 12, 8]`。

### 中文 vs 英文的差异

| 语言 | 分词难度 | 常见做法 |
|------|---------|---------|
| 英文 | 容易，按空格+标点切就行 | `text.lower().split()` 或 `nltk.word_tokenize` |
| 中文 | 难，词与词之间没有空格 | **字符级**（一个汉字一个 token）或**词级**（用 jieba 分词） |

**入门优先选「字符级」**：
- 优点：实现简单，没有 OOV（每个字都在词表里），词表小（常用汉字 ~5000）
- 缺点：序列变长（句子长度按字数算）

进阶可换 jieba 词级、或 BPE / SentencePiece（子词级，工业界主流）。

In [ ]:
# 中文：字符级分词（最简单，逐字切）
def tokenize_zh(text):
    return list(text)  # '我爱你' -> ['我', '爱', '你']

# 英文：lower + 按空格切
def tokenize_en(text):
    return text.lower().split()  # 'I love you' -> ['i', 'love', 'you']

print(tokenize_zh('我爱你'))
print(tokenize_en('I love you'))
print(tokenize_zh('今天天气真好'))
print(tokenize_en('The weather is nice today'))

## 3. 构建词表（Vocab）

词表 = `token <-> id` 的双向映射。

### 必备的 4 个特殊 token

| Token | 作用 | 典型 id |
|-------|------|--------|
| `<pad>` | 把短句填到统一长度 | 0 |
| `<sos>` | Decoder 起始信号（start of sentence） | 1 |
| `<eos>` | 序列结束信号（end of sentence） | 2 |
| `<unk>` | 未登录词（推理时遇到陌生词） | 3 |

**为什么 `<pad>` 是 0**：方便 `nn.Embedding(padding_idx=0)` 自动屏蔽 padding 位置的梯度。

In [ ]:
from collections import Counter

PAD, SOS, EOS, UNK = 0, 1, 2, 3
SPECIALS = ['<pad>', '<sos>', '<eos>', '<unk>']

class Vocab:
    def __init__(self, token_lists, min_freq=1):
        """token_lists: 二维列表 [[t1,t2,...], [t3,t4,...]]"""
        counter = Counter()
        for tokens in token_lists:
            counter.update(tokens)
        # 特殊 token 占前 4 个 id；其余按频次降序
        self.itos = list(SPECIALS) + [t for t, c in counter.most_common() if c >= min_freq]
        self.stoi = {t: i for i, t in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def encode(self, tokens):
        """token 列表 -> id 列表，未登录词用 <unk>"""
        return [self.stoi.get(t, UNK) for t in tokens]

    def decode(self, ids):
        """id 列表 -> token 列表（用于看模型输出）"""
        return [self.itos[i] for i in ids]


# 用我们的小语料构建词表
src_tokens = [tokenize_zh(zh) for zh, en in raw_pairs]
tgt_tokens = [tokenize_en(en) for zh, en in raw_pairs]

src_vocab = Vocab(src_tokens)
tgt_vocab = Vocab(tgt_tokens)

print(f'中文词表大小: {len(src_vocab)}')
print(f'英文词表大小: {len(tgt_vocab)}')
print(f"src_vocab['我'] = {src_vocab.stoi['我']}")
print(f"tgt_vocab['love'] = {tgt_vocab.stoi['love']}")
print(f"src_vocab.encode(['我','爱','你']) = {src_vocab.encode(list('我爱你'))}")

## 4. 句子 → id 序列（加 SOS/EOS）

**约定**：
- `src`：原句子 + `<eos>`，**不需要 `<sos>`**（Encoder 不需要起始信号）
- `tgt`：`<sos>` + 原句子 + `<eos>`，**两端都加**（Decoder 用 `<sos>` 起步、`<eos>` 收尾）

训练时还会拆成：
- `tgt_in = tgt[:-1]` → Decoder 输入（去掉末尾 `<eos>`）
- `tgt_out = tgt[1:]` → 计算 loss 的目标（去掉开头 `<sos>`，错位一格）

In [ ]:
def sentence_to_ids(sentence, vocab, tokenizer, add_sos=False, add_eos=True):
    ids = vocab.encode(tokenizer(sentence))
    if add_sos:
        ids = [SOS] + ids
    if add_eos:
        ids = ids + [EOS]
    return ids

src_ids = sentence_to_ids('我爱你', src_vocab, tokenize_zh, add_sos=False, add_eos=True)
tgt_ids = sentence_to_ids('I love you', tgt_vocab, tokenize_en, add_sos=True,  add_eos=True)

print('src ids:', src_ids, '->', src_vocab.decode(src_ids))
print('tgt ids:', tgt_ids, '->', tgt_vocab.decode(tgt_ids))

## 5. Padding：把不等长句子拼成矩阵

**问题**：一个 batch 里 3 句话长度可能是 4、6、9，不能直接堆成张量（张量必须矩形）。

**解法**：找出 batch 内最长那句的长度 `max_len`，所有短句在末尾补 `<pad>` 到 `max_len`。

```
原始（变长）              padding 后（max_len=9）
[5, 12, 8, 2]            [5, 12, 8, 2, 0, 0, 0, 0, 0]
[7, 3, 9, 11, 4, 2]   →  [7, 3, 9, 11, 4, 2, 0, 0, 0]
[6, 8, 1, 4, 5, 7, 9, 3, 2]  [6, 8, 1, 4, 5, 7, 9, 3, 2]
```

### 两种 padding 策略

| 策略 | 做法 | 优缺点 |
|------|------|--------|
| **全局 padding** | 所有句子补到「整个数据集最长」 | 简单，但浪费算力（短句也跟着变长） |
| **动态 padding（推荐）** | 每个 batch 内独立补到「本 batch 最长」 | 高效，是工业界标配 |

动态 padding 的关键是：**`DataLoader` 的 `collate_fn` 自定义拼 batch 的逻辑**。

In [ ]:
import torch

def pad_sequence(ids_list, pad_id=PAD):
    """ids_list: [[1,2,3],[4,5],[6,7,8,9]] -> Tensor (batch, max_len)"""
    max_len = max(len(ids) for ids in ids_list)
    padded = [ids + [pad_id] * (max_len - len(ids)) for ids in ids_list]
    return torch.tensor(padded, dtype=torch.long)

# 演示
demo = [[5, 12, 8, 2], [7, 3, 9, 11, 4, 2], [6, 8, 1, 4, 5, 7, 9, 3, 2]]
padded = pad_sequence(demo)
print(padded)
print('shape:', padded.shape)

## 6. 为什么要保存「真实长度」

Padding 后形状统一了，但 GRU **不知道哪些位置是真句子、哪些是 `<pad>`**。如果不告诉它，它会把 pad 也当输入算 hidden，污染最终的 context vector。

### 解法 1：`pack_padded_sequence`（高效，工业级）

PyTorch 提供 `nn.utils.rnn.pack_padded_sequence`，喂给 GRU 时把 padding 跳过、只算真实步。需要传入每条句子的**真实长度** `src_len`。

### 解法 2：mask（Attention 必备）

用 `mask = (src != PAD)` 生成布尔矩阵，在 Attention 计算时把 pad 位置的分数置成 `-inf`，softmax 后就是 0。

**所以 batch 一定要带 `src_len`**，下面 collate_fn 会一并产出。

## 7. Dataset + DataLoader 完整流水线

PyTorch 的标准套路：
1. **Dataset**：定义「单条样本」怎么拿（`__getitem__`）和「总共多少条」（`__len__`）
2. **collate_fn**：定义「一批样本怎么拼成 batch」（动态 padding 在这里实现）
3. **DataLoader**：负责 shuffle、batching、多进程加载

In [ ]:
from torch.utils.data import Dataset, DataLoader

class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab):
        self.pairs = pairs
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        zh, en = self.pairs[idx]
        src_ids = sentence_to_ids(zh, self.src_vocab, tokenize_zh, add_sos=False, add_eos=True)
        tgt_ids = sentence_to_ids(en, self.tgt_vocab, tokenize_en, add_sos=True,  add_eos=True)
        return src_ids, tgt_ids


def collate_fn(batch):
    """batch: [(src_ids, tgt_ids), ...]
    返回: src(B,Smax), src_len(B,), tgt(B,Tmax)
    """
    # 习惯按 src 长度降序排列（pack_padded_sequence 要求）
    batch.sort(key=lambda x: len(x[0]), reverse=True)
    src_list, tgt_list = zip(*batch)

    src_len = torch.tensor([len(s) for s in src_list], dtype=torch.long)
    src = pad_sequence(list(src_list))
    tgt = pad_sequence(list(tgt_list))
    return src, src_len, tgt


dataset = TranslationDataset(raw_pairs, src_vocab, tgt_vocab)
loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

for src, src_len, tgt in loader:
    print('src      :', src.shape, '<- (batch, src_max_len)')
    print('src_len  :', src_len, '<- 每条真实长度（含 EOS）')
    print('tgt      :', tgt.shape, '<- (batch, tgt_max_len)')
    print('src[0]   :', src[0].tolist())
    print('decode   :', src_vocab.decode(src[0].tolist()))
    print('tgt[0]   :', tgt[0].tolist())
    print('decode   :', tgt_vocab.decode(tgt[0].tolist()))
    break

## 8. 训练时怎么用这个 batch

等 U10 实现 baseline 翻译模型时，训练循环大致这样用 batch：

```python
for src, src_len, tgt in loader:
    tgt_in  = tgt[:, :-1]   # decoder 输入：<sos> ... 倒数第二个
    tgt_out = tgt[:, 1:]    # loss 目标 ：第二个 ... <eos>

    logits = model(src, src_len, tgt_in)         # (B, T-1, vocab)
    loss   = criterion(
        logits.reshape(-1, logits.size(-1)),     # (B*(T-1), vocab)
        tgt_out.reshape(-1),                     # (B*(T-1),)
    )
    # 注意 criterion = nn.CrossEntropyLoss(ignore_index=PAD) 自动忽略 pad 位置
```

## 9. 数据流总结（必须能默写）

```
原始字符串
    │  tokenizer
    ▼
token 列表  ['我','爱','你']
    │  Vocab.encode
    ▼
id 列表  [5, 12, 8]
    │  + SOS / EOS
    ▼
完整 id  [1, 5, 12, 8, 2]
    │  Dataset.__getitem__
    ▼
单样本 (src_ids, tgt_ids)
    │  DataLoader + collate_fn（动态 padding）
    ▼
batch:  src (B, Smax) | src_len (B,) | tgt (B, Tmax)
    │  Embedding
    ▼
(B, Smax, embed_dim)  ← 进 Encoder
```

## 10. 本单元小结

- **中文常用字符级**入门；英文按空格 + lower。
- 词表必有 4 个特殊 token：`<pad> <sos> <eos> <unk>`，且 `<pad>=0`。
- `src` 加 `<eos>`；`tgt` 加 `<sos>` 和 `<eos>`。
- 同一 batch 内**动态 padding**比全局 padding 高效；`collate_fn` 是关键。
- 必须连同 `src_len` 一起返回，给 `pack_padded_sequence` 用。
- `tgt_in = tgt[:,:-1]`、`tgt_out = tgt[:,1:]` 是 Seq2Seq 训练的标准错位。